# 06 — Memory across runs
Flow state belongs to one run. Memory can be recalled in a later run.
Explicit scopes keep Customer A's preference out of Customer B's context.



## Problem

A new run does not automatically know a customer's saved constraint.


## Concept

CrewAI Memory recalls retained facts; explicit scopes separate customers.

Memory retains selected facts across invocations. A scoped recall narrows what is eligible to be returned, so customer identity must be chosen by the application, never guessed from the prompt.


## Abstraction

customer A/B facts → scoped store → scoped recall in a later run. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
customer A fact ──▶ /customer/customer-a ──┐
                                           ├──▶ scoped recall ──▶ next run
customer B fact ──▶ /customer/customer-b ──┘
                  (B's facts must not enter A's context)
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Create Memory and save two customers' facts in distinct scopes

Create Memory and save two customers' facts in distinct scopes.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage
from crewai import Memory

from workshop.common import embedder_config
from workshop.memory import customer_scope

memory = Memory(embedder=embedder_config())
memory.remember(
    "Customer A requires private VPC deployment.",
    scope=customer_scope("customer-a"),
    source="customer:customer-a",
)
memory.remember(
    "Customer B prefers a managed SaaS pilot.",
    scope=customer_scope("customer-b"),
    source="customer:customer-b",
)



## Recall from a customer scope
Inspect both results. A result from the wrong customer indicates a serious
isolation problem, not just a weak answer.



## Step 2 — Recall each scope independently and inspect the tree for isolation

Recall each scope independently and inspect the tree for isolation.


In [ ]:
for customer in ("customer-a", "customer-b"):
    matches = memory.recall("deployment preference", scope=customer_scope(customer), limit=3)
    print(customer, [match.record.content for match in matches])
print("Memory tree:\n", memory.tree())


## Execution flow

customer A/B facts → scoped store → scoped recall in a later run. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

Using a global scope leaks one customer's fact into another customer's run. The scope identifier is a security boundary, not a personalization convenience.


## Production implication

Define what may be retained, for how long, and how it is deleted. Test tenant isolation with negative cases, not just successful recall.


## Your turn

Search both scopes for deployment preferences and inspect the memory tree.

**Checkpoint:** A's VPC preference does not appear in B's recall.
